In [ ]:
# api key : ak_2YY3kz0rK5xO5Fc3Z56BG9FS5n005

In [ ]:
import requests

url = "https://api.longcat.chat/openai/v1/chat/completions"
headers = {
    "Authorization": "Bearer your_api_key_here",
    "Content-Type": "application/json"
}

data = {
    "model": "LongCat-Flash-Chat",
    "messages": [
        {"role": "user", "content": "Hello, please introduce yourself."}
    ],
    "max_tokens": 1000,
    "temperature": 0.7
}

response = requests.post(url, headers=headers, json=data)
print(response.json())


{'id': '902a4158a93549dc8f93be44480043a5', 'object': 'chat.completion', 'created': 1773823784, 'model': 'longcat-flash-chatai-api', 'usage': {'completion_tokens': 59, 'prompt_tokens': 17, 'total_tokens': 76, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'cached_tokens': 0}, 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "Hello! I'm an AI language model developed by Meituan. I can assist you with a wide range of tasks, such as answering questions, providing explanations, helping with writing, offering suggestions, and more. Feel free to ask me anything—I'm here to help! 😊"}, 'finish_reason': 'stop', 'matched_stop': 2, 'logprobs': None}]}


In [2]:
pip install datasets


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:

from datasets import load_dataset

# Stream the dataset and take the first 25k rows
ds_stream = load_dataset("nvidia/Nemotron-Personas-India", split='en_IN', streaming=True)
first_25k = ds_stream.take(25000)

# Convert to a standard list or pandas DataFrame if needed
import pandas as pd
df = pd.DataFrame(list(first_25k))

c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# df

In [2]:

PROMPT_TEMPLATE_PRESCRIPTION = """
You are a data augmentation assistant for information extraction models on medical prescriptions.
Your task is to generate synthetic variations of a given OCR medical record.

INPUT:
You will receive a medical record in text strings.

GOAL:
Generate 3 new variations of the record.

AUGMENTATION RULES:

1. Patient Information

* Patient demographic information will be provided from an external persona dataset (NVIDIA Nemotron Personas India).
* Use the provided persona values to fill or replace the patient-related fields such as:
  patient_name, patient_age, patient_sex/gender, and optionally patient_location or city.
* Do NOT invent random names or demographics. Only use the persona information when available.

2. Medicine Variations

* Modify medicine related fields to create realistic prescription diversity.
* You may change medicine names, dosage amounts, or instructions while keeping them medically plausible.
  Examples of variations include:
* changing dosage values (10 mg → 25 mg → 50 mg)
* changing instruction phrases (Before meals, After meals, Once daily, Twice daily)
* replacing a medicine with another common medicine.

3. Structural Changes

* Shuffle the order of fields.
* Remove 1–3 fields randomly (for example patient_age, clinic_address, signature).
* Sometimes add a new optional field like patient_sex or patient_location.
for example, if the original record does not have patient_sex, you can add it in one of the variations using the persona data.
consider this .'BioSphere  Advanced  Pathology  LABORATORY  TEST  REPORT  PatientID  368599  Collected  2025-05-27  Total  WBC:308.07fL  Potassium-187.65  fL  Glucose  Random(196.63cells/cumm)  Serum  Creatinine(322.21cells/cumm)  Hemoglobin-370.55  %  Blood  Urea:397.53%  Lab  Internal  QC  OK '
here you can remove the laboratory name, or the patient id, or the collected date, or the total wbc, or the potassium, or the glucose, or the serum creatinine, or the hemoglobin, or the blood urea, or the lab internal qc. you can also change the values of these fields. you can also add new fields like patient sex or patient location using the persona data.
and you can change values and shuffle the order of these fields.

4. ADDITION OF random senetences AND OCR ERRORS
   Introduce random words or phrases that could realistically appear in the context of a medical record. Additionally, simulate OCR errors by altering characters in the text.as generally the ocr is a bit long and not that short as we have considered 

5. Formatting Variations
   Change formatting such as:

Age
35
35 yrs
35 years

Date
2024-12-16
16/12/2024
Dec 16 2024

Doctor Name
Dr. A. Smith
Dr A Smith
Dr.A.Smith

Gender
Male
M
Sex: Male

6. Important Constraints

* Do NOT explain anything.
* Only output the generated records.
* Ensure the 3 generated records are structurally different from each other.

7. CHANGE the name of labs and doctors to create more diversity.
lets say if the original record has a lab name "BioSphere Advanced Pathology LABORATORY TEST REPORT" you can change it to "HealthFirst Diagnostic Center" or "MediLab Pathology Services" in the variations. Similarly, for doctor names, if the original record mentions "Dr. A. Smith", you can create variations like "Dr. R. Kumar" or "Dr. S. Patel". This will help in creating more diverse and realistic synthetic records.

8. INTRODUCTION OF TYPOS AND JUSTIFIED OCR ERRORS
introduce typos in the variations to mimic real-world OCR errors. For example, "WBC" could be written as "W8C" or "Glucose" could be "Glocose". This will help in making the augmented data more robust for training OCR models.

OUTPUT FORMAT:

Return ONLY a JSON array of 3 strings.

Example:
[
"",
"",
""
]

INPUT RECORD:
{text}

PERSONA DATA:
{persona}
"""


In [16]:

PROMPT_TEMPLATE_NON_PRESCRIPTION = """
You are a data augmentation assistant for OCR-based information extraction models.

Your task is to generate synthetic variations of laboratory reports / diagnostic records (non-prescription data).

INPUT:
You will receive a medical record in text strings.

GOAL:
Generate 3 new variations of the record.

AUGMENTATION RULES:

1. Report & Entity Variations

* Change laboratory / hospital names to realistic alternatives
  (e.g., "AIG Hospitals Laboratory" → "HealthFirst Diagnostics", "MediCare Labs", "PrimePath Labs")
* Modify or replace **report titles**
  (e.g., "LABORATORY TEST REPORT" → "Diagnostic Report", "Investigation Summary")


 2. Test Parameter Variations

* Modify test values while keeping formats realistic:

  * Change numeric values (e.g., 134.66 → 120.45 → 298.12)
  * Slightly alter units (IU/L, g/dL, mmol/L, %, fL, cells/cumm, sec)
* You may:

  * Remove some test parameters
  * Add new common lab parameters such as:

    * Platelet Count
    * RBC
    * Sodium / Potassium
    * Urea
    * Creatinine
    * Hemoglobin
    * CRP
* Keep values noisy/unstructured like OCR outputs

 3. Structural Changes

* Shuffle field order randomly
* Remove 1–4 fields (e.g., PatientID, Collected date, specific tests)
* Optionally add new fields:

  * Patient Name
  * Age
  * Gender / Sex
  * Location
* Add or remove sections like:

  * "Medication Advice"
  * "Lab Internal QC OK"
  * "End Of Report"

  
4. ADDITION OF random senetences AND OCR ERRORS
   Introduce random words or phrases that could realistically appear in the context of a medical record. Additionally, simulate OCR errors by altering characters in the text.as generally the ocr is a bit long and not that short as we have considered 

5. Formatting Variations

Introduce formatting diversity:

Dates:

* 2025-02-11
* 11/02/2025
* Feb 11 2025

PatientID:

* PatientID 257136
* ID: 257136
* PID-257136

Test entries:

* RDW-134.66 IU/L
* RDW : 134.66 IU/L
* RDW(134.66IU/L)

Spacing / casing:

* RANDOM CAPS
* inconsistent spacing
* merged words

6. OCR Noise & Typos (IMPORTANT)

Introduce realistic OCR errors:

* Character swaps:
 * O ↔ 0
  * I ↔ 1
  * B ↔ 8
* Example:
 * WBC → W8C
  * Sodium → Sodiurn
  * Glucose → Glocose
* Broken tokens:

  * "Creatinine" → "Creatin ine"
  * "Hemoglobin" → "Hem0globin"

 7. Optional Medication Section

* Sometimes include a Medication Advice section
* Modify medicine names and dosage patterns:

  * Tab Paracetamol 1-0-1
  * Cap Azithromycin 0-1-0
* Or remove this section entirely

 8. Important Constraints

* Do NOT explain anything
* Output ONLY the generated records
* Ensure all 3 outputs are structurally different
* Keep text in raw OCR-like format (no JSON structure inside strings).

 OUTPUT FORMAT:

Return ONLY a JSON array of 3 strings:

[
"",
"",
""
]

---

INPUT RECORD:
{text}

PERSONA DATA:
{persona}

"""


In [17]:
def make_json_safe(obj):
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    elif hasattr(obj, "item"):  # numpy types
        return obj.item()
    else:
        return obj

In [7]:
import random

def sample_persona(df):
    row = df.sample(1).iloc[0]

    return {
        "name": row.get("name", ""),
        "age": row.get("age", ""),
        "gender": row.get("gender", ""),
        "city": row.get("city", "")
    }

In [18]:
persona = make_json_safe(sample_persona(df))

In [ ]:
import requests
import json

url = "https://api.longcat.chat/openai/v1/chat/completions"

headers = {
    "Authorization": "Bearer your_api_key_here",
    "Content-Type": "application/json"
}

def call_llm(prompt):
    data = {
        "model": "LongCat-Flash-Lite",
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 1000,
        "temperature": 0.85
    }

    response = requests.post(url, headers=headers, json=data)
    # requests.post(..., timeout=30)

    return response.json()

In [25]:
prompt_non_prescription = PROMPT_TEMPLATE_NON_PRESCRIPTION.format(
                text="hi",
                persona=json.dumps(persona, indent=2)
            )
call_llm("HI")

{'id': '5c8a9aeb7f7a451bab3132dcc99860ca',
 'object': 'chat.completion',
 'created': 1774002916,
 'model': 'longcat-flash-3b-all-quant-0203-eagle3',
 'usage': {'completion_tokens': 12,
  'prompt_tokens': 3,
  'total_tokens': 15,
  'cache_write_tokens': 0,
  'cache_read_tokens': 0,
  'input_tokens': 0,
  'output_tokens': 0,
  'cached_tokens': 0},
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Hello! How can I help you today? 😊'},
   'finish_reason': 'stop',
   'matched_stop': 2,
   'logprobs': None}]}

In [26]:
def augment_text(text, df):

    # persona = make_json_safe(sample_persona(df))

   

    

    try:
        output = response["choices"][0]["message"]["content"]
        variations = json.loads(output)
    except:
        variations = []

    return variations

In [27]:
import pandas as pd

original_df = pd.read_csv("C:\\Users\\user\\Downloads\\140924_combined_reports_prescriptions_7600.csv")

In [28]:
print(original_df.columns)

Index(['text', 'label', 'filename'], dtype='object')


In [12]:
# augmented_data = []

# for text in original_df["text"]:

#     augmented_data.append(text)  # original

#     variations = augment_text(text, df)

#     augmented_data.extend(variations)

In [29]:
!pip install tqdm rich


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
# import json
# import pandas as pd
# from tqdm import tqdm
# from rich.console import Console
# from rich.panel import Panel
# from rich.pretty import Pretty

# console = Console()

# augmented_rows = []

# # progress bar
# for idx, row in tqdm(original_df.iterrows(), total=len(original_df), desc="Augmenting Data"):

#     text = row["text"]
#     label = row.get("label", None)
#     filename = row.get("filename", None)

#     if pd.isna(text) or text.strip() == "":
#         continue

#     # sample persona
#     persona = make_json_safe(sample_persona(df))

#     # build prompts
#     prompt_prescription = PROMPT_TEMPLATE_PRESCRIPTION.format(
#         text=text,
#         persona=json.dumps(persona, indent=2)
#     )

#     prompt_non_prescription = PROMPT_TEMPLATE_NON_PRESCRIPTION.format(
#         text=text,
#         persona=json.dumps(persona, indent=2)
#     )

#     try:
#         if label == 1:
#             response = call_llm(prompt_non_prescription)
#             prompt_type = "NON-PRESCRIPTION"
#         else:
#             response = call_llm(prompt_prescription)
#             prompt_type = "PRESCRIPTION"

#         output = response["choices"][0]["message"]["content"]

#         # # 🎯 Beautiful logging
#         # console.print(Panel.fit(f"[bold cyan]Row {idx} | {prompt_type}[/bold cyan]"))

#         # console.print(Panel(
#         #     text,
#         #     title="[bold green]Original Text[/bold green]",
#         #     border_style="green"
#         # ))

#         variations = json.loads(output)

#         # ✅ log original
#         augmented_rows.append({
#             "original_text": text,
#             "augmented_text": text,
#             "label": label,
#             "filename": filename,
#             "persona": json.dumps(persona),
#             "status": "original"
#         })

#         # ✅ log augmented
#         for i, v in enumerate(variations):
#             # console.print(Panel(
#             #     v,
#             #     title=f"[bold yellow]Augmented #{i+1}[/bold yellow]",
#             #     border_style="yellow"
#             # ))

#             augmented_rows.append({
#                 "original_text": text,
#                 "augmented_text": v,
#                 "label": label,
#                 "filename": filename,
#                 "persona": json.dumps(persona),
#                 "raw_response": output,
#                 "status": "augmented"
#             })

#     except Exception as e:

#         console.print(Panel(
#             f"[bold red]Error:[/bold red] {str(e)}",
#             title=f"[red]Row {idx} Failed[/red]",
#             border_style="red"
#         ))

#         augmented_rows.append({
#             "original_text": text,
#             "augmented_text": "",
#             "label": label,
#             "filename": filename,
#             "persona": json.dumps(persona),
#             "status": f"failed: {str(e)}"
#         })

import json
import pandas as pd
from tqdm import tqdm
from rich.console import Console
from rich.panel import Panel
from concurrent.futures import ThreadPoolExecutor, as_completed

console = Console()

augmented_rows = []

MAX_WORKERS = 20


def process_row(idx, row):
    try:
        text = row["text"]
        label = row.get("label", None)
        filename = row.get("filename", None)

        if pd.isna(text) or text.strip() == "":
            return []

        # sample persona
        persona = make_json_safe(sample_persona(df))

        # build prompts
        prompt_prescription = PROMPT_TEMPLATE_PRESCRIPTION.format(
            text=text,
            persona=json.dumps(persona, indent=2)
        )

        prompt_non_prescription = PROMPT_TEMPLATE_NON_PRESCRIPTION.format(
            text=text,
            persona=json.dumps(persona, indent=2)
        )

        if label == 1:
            response = call_llm(prompt_non_prescription)
        else:
            response = call_llm(prompt_prescription)

        output = response["choices"][0]["message"]["content"]

        variations = json.loads(output)

        local_rows = []

        # ✅ original
        local_rows.append({
            "original_text": text,
            "augmented_text": text,
            "label": label,
            "filename": filename,
            "persona": json.dumps(persona),
            "status": "original"
        })

        # ✅ augmented
        for v in variations:
            local_rows.append({
                "original_text": text,
                "augmented_text": v,
                "label": label,
                "filename": filename,
                "persona": json.dumps(persona),
                "raw_response": output,
                "status": "augmented"
            })

        return local_rows

    except Exception as e:
        print(f"Error processing row {idx}: {str(e)}")

        return [{
            "original_text": row.get("text", ""),
            "augmented_text": "",
            "label": row.get("label", None),
            "filename": row.get("filename", None),
            "persona": "",
            "status": f"failed: {str(e)}"
        }]


# 🚀 Run with ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = [
        executor.submit(process_row, idx, row)
        for idx, row in original_df.iterrows()
    ]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Augmenting Data"):
        result = future.result()
        if result:
            augmented_rows.extend(result)

Augmenting Data:   0%|          | 21/7608 [00:05<34:39,  3.65it/s] 

Error processing row 22: Expecting value: line 5 column 1 (char 934)


Augmenting Data:   0%|          | 27/7608 [00:05<19:39,  6.43it/s]

Error processing row 18: Expecting value: line 5 column 1 (char 1135)


Augmenting Data:   1%|          | 40/7608 [00:09<21:08,  5.97it/s]

...

Augmenting Data: 100%|█████████▉| 7570/7608 [28:03<00:05,  6.60it/s]

Error processing row 7573: Expecting value: line 5 column 1 (char 444)


Augmenting Data: 100%|█████████▉| 7586/7608 [28:06<00:03,  5.80it/s]

Error processing row 7583: Unterminated string starting at: line 3 column 1 (char 195)


Augmenting Data: 100%|██████████| 7608/7608 [28:15<00:00,  4.49it/s]


In [31]:
# import pandas as pd

# aug_df = pd.DataFrame(augmented_rows)

# aug_df.to_csv("C:\\Users\\user\\Documents\\augmented_dataset.csv", index=False)

In [32]:
# aug_df.to_csv("C:\\Users\\user\\Documents\\augmented_dataset123.csv", index=False)

In [33]:

PROMPT_TEMPLATE_NON_PRESCRIPTION = """
You are a data augmentation assistant for OCR-based information extraction models.

Your task is to generate synthetic variations of laboratory reports / diagnostic records (non-prescription data).

INPUT:
You will receive a medical record in text strings.

GOAL:
Generate 3 new variations of the record.

AUGMENTATION RULES:

1. Report & Entity Variations

* Change laboratory / hospital names to realistic alternatives
  (e.g., "AIG Hospitals Laboratory" → "HealthFirst Diagnostics", "MediCare Labs", "PrimePath Labs")
* Modify or replace **report titles**
  (e.g., "LABORATORY TEST REPORT" → "Diagnostic Report", "Investigation Summary")


 2. Test Parameter Variations

* Modify test values while keeping formats realistic:

  * Change numeric values (e.g., 134.66 → 120.45 → 298.12)
  * Slightly alter units (IU/L, g/dL, mmol/L, %, fL, cells/cumm, sec)
* You may:

  * Remove some test parameters
  * Add new common lab parameters such as:

    * Platelet Count
    * RBC
    * Sodium / Potassium
    * Urea
    * Creatinine
    * Hemoglobin
    * CRP
* Keep values noisy/unstructured like OCR outputs

 3. Structural Changes

* Shuffle field order randomly
* Remove 1–4 fields (e.g., PatientID, Collected date, specific tests)
* Optionally add new fields:

  * Patient Name
  * Age
  * Gender / Sex
  * Location
* Add or remove sections like:

  * "Medication Advice"
  * "Lab Internal QC OK"
  * "End Of Report"

4. Formatting Variations

Introduce formatting diversity:

Dates:

* 2025-02-11
* 11/02/2025
* Feb 11 2025

PatientID:

* PatientID 257136
* ID: 257136
* PID-257136

Test entries:

* RDW-134.66 IU/L
* RDW : 134.66 IU/L
* RDW(134.66IU/L)

Spacing / casing:

* RANDOM CAPS
* inconsistent spacing
* merged words

5. OCR Noise & Typos (IMPORTANT)

Introduce realistic OCR errors:

* Character swaps:
 * O ↔ 0
  * I ↔ 1
  * B ↔ 8
* Example:
 * WBC → W8C
  * Sodium → Sodiurn
  * Glucose → Glocose
* Broken tokens:

  * "Creatinine" → "Creatin ine"
  * "Hemoglobin" → "Hem0globin"

 6. Optional Medication Section

* Sometimes include a Medication Advice section
* Modify medicine names and dosage patterns:

  * Tab Paracetamol 1-0-1
  * Cap Azithromycin 0-1-0
* Or remove this section entirely

 7. Important Constraints

* Do NOT explain anything
* Output ONLY the generated records
* Ensure all 3 outputs are structurally different
* Keep text in raw OCR-like format (no JSON structure inside strings).

 OUTPUT FORMAT:

Return ONLY a JSON array of 3 strings:

[
"",
"",
""
]

---

INPUT RECORD:
{text}

PERSONA DATA:
{persona}

"""


In [34]:
PROMPT_TEMPLATE_MEDICAL_OTHERS = """
You are a data augmentation assistant.

Given an input text that may look like a medical report, lab result, or prescription, your task is to transform it into formats that DO NOT resemble structured medical documents.

Your goal is to generate outputs that belong to the "other" class in a classification dataset.

### Instructions:
1. Convert the input into ONE of the following styles (randomly choose):
   - Casual conversation (2–3 people talking)
   - WhatsApp/chat messages
   - Story or narrative paragraph
   - General discussion or explanation
   - Blog-style writing
   - Question-answer dialogue
   - Personal note or diary entry

2. IMPORTANT:
   - Do NOT preserve structured medical formatting (no bullet lists like prescriptions, no "Name:", "Age:", etc.)
   - Do NOT keep it looking like a report
   - You MAY keep or slightly distort the meaning, but presentation must change completely
   - You MAY omit, reorder, or generalize details
   - You MAY introduce informal language, filler words, or conversational tone


3. Optional Noise Injection:
   - Add small talk, interruptions, or irrelevant details
   - Introduce ambiguity or partial information
   - Slightly alter names, numbers, or entities

4. - Do NOT preserve structured medical formatting  
  (no "Name:", "Age:", bullet lists, tables, or prescription layout)
- Break ordering of information (shuffle, drop, merge details)
- Use natural, human-like phrasing
- You MAY distort or generalize the meaning

5.  OCR Noise + Typo Injection (VERY IMPORTANT):

Introduce realistic OCR errors and human typing mistakes:

- Character substitutions:
  - o → 0, l → 1, e → c, a → @, s → 5
- Missing or extra spaces:
  - "bloodpressure" / "blood  pressure"
- Broken or merged words:
  - "medicationtaken" / "medi cation"
- Random capitalization:
  - "pAtient", "DOcTor"
- Spelling mistakes:
  - "fever" → "fevr", "tablet" → "tablct"
- Punctuation noise:
  - extra commas, missing full stops, "??", "..."
- Slight numeric distortions:
  - 500 → 50O, 25 → 2S
- Partial truncation:
  - cut words midway ("prescrip...", "medic...")

 Do NOT overdo noise — keep text readable but imperfect (like OCR output).

6. Output must:
   - Be natural and human-like
   - Clearly NOT resemble a medical document
   - Be suitable for classification as "other"

### Input:
{text}

OUTPUT FORMAT:


Return ONLY a JSON array of string:

[
"",

]


"""

In [35]:
# prompt_other = PROMPT_TEMPLATE_MEDICAL_OTHERS.format(
#             text=text ,
#         )

In [36]:
# import json
# import pandas as pd
# from tqdm import tqdm
# from rich.console import Console
# from rich.panel import Panel
# from concurrent.futures import ThreadPoolExecutor, as_completed

# console = Console()

# augmented_rows = []

# MAX_WORKERS = 7


# def process_row(idx, row):
#     try:
#         text = row["text"]
#         label = row.get("label", None)
#         filename = row.get("filename", None)

#         if pd.isna(text) or text.strip() == "":
#             return []

#         # sample persona
#         persona = make_json_safe(sample_persona(df))

#         # build prompts
#         # prompt_prescription = PROMPT_TEMPLATE_PRESCRIPTION.format(
#         #     text=text,
#         #     persona=json.dumps(persona, indent=2)
#         # )

#         prompt_other = PROMPT_TEMPLATE_MEDICAL_OTHERS.format(
#             text=text 
#         )

     
           
        
#         response = call_llm(prompt_other)

#         output = response["choices"][0]["message"]["content"]
        
#         # print( "llm_output" , output) 
#         variations = json.loads(output)

#         local_rows = []

#         # ✅ original
#         local_rows.append({
#             "original_text": text,
#             "augmented_text": text,
#             "label": label,
#             "filename": filename,
#             "persona": json.dumps(persona),
#             "status": "original"
#         })

#         # ✅ augmented
#         for v in variations:
#             local_rows.append({
#                 "original_text": text,
#                 "augmented_text": v,
#                 "label": label,
#                 "filename": filename,
#                 "persona": json.dumps(persona),
#                 "raw_response": output,
#                 "status": "augmented"
#             })

#         return local_rows

#     except Exception as e:
#         print(f"Error processing row {idx}: {str(e)}")

#         return [{
#             "original_text": row.get("text", ""),
#             "augmented_text": "",
#             "label": row.get("label", None),
#             "filename": row.get("filename", None),
#             "persona": "",
#             "status": f"failed: {str(e)}"
#         }]


# # 🚀 Run with ThreadPoolExecutor
# with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

#     futures = [
#         executor.submit(process_row, idx, row)
#         for idx, row in original_df.iterrows()
#     ]

#     for future in tqdm(as_completed(futures), total=len(futures), desc="Augmenting Data"):
#         result = future.result()
#         if result:
#             augmented_rows.extend(result)

In [ ]:
import json
import os
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
import time

# --- CONSTANTS ---
INPUT_CSV = "C:\\Users\\user\\Downloads\\140924_combined_reports_prescriptions_7600.csv"       # Your extracted text from the Aadhar OCR
OUTPUT_JSONL = "augmented_data.jsonl" # This will save results line-by-line
MAX_WORKERS = 5                     # Don't go too high or the API will hang
TIMEOUT = 45                        # Seconds to wait for LLM before giving up

# Thread lock to prevent multiple threads from writing to the file at the same time
file_lock = threading.Lock()

def save_result(data_list):
    """Safely appends results to the JSONL file."""
    with file_lock:
        with open(OUTPUT_JSONL, "a", encoding="utf-8") as f:
            for entry in data_list:
                f.write(json.dumps(entry) + "\n")

def get_processed_files():
    """Checks the output file to see which filenames are already finished."""
    if not os.path.exists(OUTPUT_JSONL):
        return set()
    
    processed = set()
    with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            try:
                data = json.loads(line)
                processed.add(data["filename"])
            except:
                continue
    return processed

def process_row(idx, row):
    """Your logic for one row."""
    try:
        text = row["text"]
        label = row.get("label", "N/A")
        filename = row.get("filename", f"row_{idx}")

        if pd.isna(text) or text.strip() == "":
            return None

        # Build your prompt
        prompt_other = PROMPT_TEMPLATE_MEDICAL_OTHERS.format(text=text)

        # Call LLM (Make sure your call_llm function has an internal timeout!)
        # Example: response = client.chat.completions.create(..., timeout=30)
        response = call_llm(prompt_other) 
        
        output = response["choices"][0]["message"]["content"]
        variations = json.loads(output)

        local_rows = []
        # Store Original
        local_rows.append({
            "filename": filename,
            "original_text": text,
            "augmented_text": text,
            "label": label,
            "status": "original"
        })
        # Store Augmented
        for v in variations:
            local_rows.append({
                "filename": filename,
                "original_text": text,
                "augmented_text": v,
                "label": label,
                "status": "augmented"
            })
        return local_rows

    except Exception as e:
        # Logging errors to console so you see why it failed
        print(f"\n[ERROR] Row {idx} ({filename}): {str(e)}")
        return None

def main():
    # 1. Load Data
    df = pd.read_csv(INPUT_CSV)
    
    # 2. Check for Checkpoints (Resume Logic)
    processed_filenames = get_processed_files()
    print(f"Total rows: {len(df)}")
    print(f"Already processed: {len(processed_filenames)}")
    
    # Filter out rows already in the JSONL
    pending_df = df[~df['filename'].isin(processed_filenames)]
    print(f"Rows remaining: {len(pending_df)}")

    if pending_df.empty:
        print("Everything is already processed!")
        return

    # 3. Start Multi-threaded Processing
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Map futures to filenames
        future_to_row = {
            executor.submit(process_row, idx, row): row['filename'] 
            for idx, row in pending_df.iterrows()
        }

        # tqdm for progress bar
        for future in tqdm(as_completed(future_to_row), total=len(future_to_row), desc="Processing"):
            filename = future_to_row[future]
            try:
                # We add a timeout here. If the thread hangs for > 60s, it raises TimeoutError
                result = future.result(timeout=TIMEOUT)
                if result:
                    # 4. SAVE SIMULTANEOUSLY TO DISK
                    save_result(result)
            except Exception as e:
                print(f"\n[CRITICAL] {filename} timed out or failed: {e}")

    print(f"\nDone! Results saved to {OUTPUT_JSONL}")

if __name__ == "__main__":
    main()

Total rows: 7608
Already processed: 0
Rows remaining: 7608


Processing:   0%|          | 4/7608 [00:03<1:59:23,  1.06it/s]


[ERROR] Row 2 (report_0003.txt): Unterminated string starting at: line 8 column 1 (char 2177)


Processing:   0%|          | 11/7608 [00:07<1:07:55,  1.86it/s]


[ERROR] Row 11 (report_0012.txt): Invalid control character at: line 2 column 226 (char 227)


...


[ERROR] Row 4334 (report_aug_extra_526.png): Expecting ',' delimiter: line 8 column 345 (char 2558)


Processing:  57%|█████▋    | 4338/7608 [1:53:09<2:48:39,  3.09s/it]


[ERROR] Row 4341 (report_aug_extra_533.png): Expecting value: line 3 column 1 (char 381)


Processing:  57%|█████▋    | 4338/7608 [1:53:14<1:25:21,  1.57s/it]



[ERROR] Row 4343 (report_aug_extra_535.png): Expecting value: line 3 column 1 (char 336)

[ERROR] Row 4344 (report_aug_extra_536.png): Expecting value: line 3 column 1 (char 397)


In [1]:
import pandas as pd

aug_df = pd.DataFrame(augmented_rows)

NameError: name 'augmented_rows' is not defined

In [ ]:
aug_df.to_csv("C:\\Users\\user\\Documents\\augmented_dataset_conversation_other.csv", index=False)

In [ ]:
import json
import pandas as pd
from tqdm import tqdm
from rich.console import Console
from rich.panel import Panel
from concurrent.futures import ThreadPoolExecutor, as_completed

console = Console()

augmented_rows1 = []

MAX_WORKERS = 20


def process_row(idx, row):
    try:
        text = row["text"]
        label = row.get("label", None)
        filename = row.get("filename", None)

        if pd.isna(text) or text.strip() == "":
            return []

        # sample persona
        persona = make_json_safe(sample_persona(df))

        # build prompts
        # prompt_prescription = PROMPT_TEMPLATE_PRESCRIPTION.format(
        #     text=text,
        #     persona=json.dumps(persona, indent=2)
        # )

        prompt_other = PROMPT_TEMPLATE_MEDICAL_OTHERS.format(
            text=text 
        )

     
           
        
        response = call_llm(prompt_other)

        output = response["choices"][0]["message"]["content"]
        
        # print( "llm_output" , output) 
        variations = json.loads(output)

        local_rows = []

        # ✅ original
        local_rows.append({
            "original_text": text,
            "augmented_text": text,
            "label": label,
            "filename": filename,
            "persona": json.dumps(persona),
            "status": "original"
        })

        # ✅ augmented
        for v in variations:
            local_rows.append({
                "original_text": text,
                "augmented_text": v,
                "label": label,
                "filename": filename,
                "persona": json.dumps(persona),
                "raw_response": output,
                "status": "augmented"
            })

        return local_rows

    except Exception as e:
        print(f"Error processing row {idx}: {str(e)}")

        return [{
            "original_text": row.get("text", ""),
            "augmented_text": "",
            "label": row.get("label", None),
            "filename": row.get("filename", None),
            "persona": "",
            "status": f"failed: {str(e)}"
        }]


# 🚀 Run with ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = [
        executor.submit(process_row, idx, row)
        for idx, row in original_df.tail(3000).iterrows()
    ]

    for future in tqdm(as_completed(futures), total=len(futures), desc="Augmenting Data"):
        result = future.result()
        if result:
            augmented_rows1.extend(result)

In [ ]:
import pandas as pd

aug_df = pd.DataFrame(augmented_rows1)

In [ ]:
aug_df.to_csv("C:\\Users\\user\\Documents\\augmented_dataset_conversation_other_bottom.csv", index=False)